In [10]:
import pandas as pd
import numpy as np
import random
import os

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import AutoTokenizer, AutoModel
import faiss
from tqdm import tqdm
import json
from pathlib import Path

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

ARTIFACTS_DIR = Path("../artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

In [11]:
transactions = pd.read_csv('drive/MyDrive/transactions_features.csv', parse_dates=['t_dat'])
articles = pd.read_csv('drive/MyDrive/articles_features.csv')

max_date = transactions['t_dat'].max()
test_start = max_date - pd.Timedelta(days=7)
val_start = test_start - pd.Timedelta(days=7)

train_data = transactions[transactions['t_dat'] < val_start].copy()
val_data = transactions[(transactions['t_dat'] >= val_start) & (transactions['t_dat'] < test_start)].copy()
test_data = transactions[transactions['t_dat'] >= test_start].copy()

In [12]:
text_columns = [
    'prod_name', 'product_type_name', 'product_group_name',
    'graphical_appearance_name', 'colour_group_name',
    'department_name', 'index_name', 'detail_desc'
]

for col in text_columns:
    articles[col] = articles[col].fillna('')

articles['text_description'] = (
    articles['prod_name'].astype(str) + ". " +
    "Category: " + articles['product_group_name'].astype(str) + " - " + articles['product_type_name'].astype(str) + ". " +
    "Style: " + articles['graphical_appearance_name'].astype(str) + ", Color: " + articles['colour_group_name'].astype(str) + ". " +
    "Department: " + articles['department_name'].astype(str) + " (" + articles['index_name'].astype(str) + "). " +
    "Description: " + articles['detail_desc'].astype(str)
)

print(articles['text_description'].iloc[0])

Strap top. Category: Garment Upper body - Vest top. Style: Solid, Color: White. Department: Jersey Basic (Ladieswear). Description: Jersey top with narrow shoulder straps.


In [13]:
model_name = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.74M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(83828, 312, padding_idx=0)
    (position_embeddings): Embedding(2048, 312)
    (token_type_embeddings): Embedding(2, 312)
    (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-2): 3 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=312, out_features=312, bias=True)
            (key): Linear(in_features=312, out_features=312, bias=True)
            (value): Linear(in_features=312, out_features=312, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=312, out_features=312, bias=True)
            (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
   

In [14]:
def extract_bert_embeddings(text_list, batch_size=256):
    all_embeddings = []
    for i in tqdm(range(0, len(text_list), batch_size), desc="BERT Inference"):
        batch_texts = text_list[i:i+batch_size]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()

        all_embeddings.append(batch_embeddings)

    return np.vstack(all_embeddings)

article_texts = articles['text_description'].tolist()
item_embeddings = extract_bert_embeddings(article_texts, batch_size=512)
print(item_embeddings.shape)

BERT Inference: 100%|██████████| 58/58 [00:38<00:00,  1.52it/s]

(29237, 312)


In [15]:
dimension = item_embeddings.shape[1]

item_embeddings_norm = item_embeddings.copy()
faiss.normalize_L2(item_embeddings_norm)

index = faiss.IndexFlatIP(dimension)
index.add(item_embeddings_norm)

item_id_to_idx = {idx: i for i, idx in enumerate(articles['article_id'].values)}
idx_to_item_id = {i: idx for i, idx in enumerate(articles['article_id'].values)}

train_data['item_idx'] = train_data['article_id'].map(item_id_to_idx)
train_data = train_data.dropna(subset=['item_idx'])
train_data['item_idx'] = train_data['item_idx'].astype(int)

user_history = train_data.groupby('customer_id')['item_idx'].apply(list).to_dict()

In [16]:
def create_query_embeddings(target_users):
    query_embs = []
    valid_users = []

    for user in target_users:
        if user in user_history:
            user_item_indices = user_history[user]
            user_profile_vector = item_embeddings[user_item_indices].mean(axis=0)
            query_embs.append(user_profile_vector)
            valid_users.append(user)

    query_embs = np.array(query_embs).astype('float32')
    if len(query_embs) > 0:
        faiss.normalize_L2(query_embs)

    return query_embs, valid_users

In [17]:
def calculate_metrics(actual_dict, predicted_dict, k):
    precisions, recalls = [], []
    for user, actual_items in actual_dict.items():
        if user not in predicted_dict:
            continue
        actual_set = set(actual_items)
        predicted_set = set(predicted_dict[user][:k])
        hits = len(actual_set & predicted_set)
        precisions.append(hits / k)
        recalls.append(hits / len(actual_set) if len(actual_set) > 0 else 0)

    if not precisions:
        return 0.0, 0.0
    return np.mean(precisions), np.mean(recalls)

def evaluate_semantic_search(target_data, split_name):
    actual_purchases = target_data.groupby('customer_id')['article_id'].apply(list).to_dict()
    target_users = list(actual_purchases.keys())

    query_embeddings, valid_users = create_query_embeddings(target_users)

    k = 20
    Distances, Indices = index.search(query_embeddings, k)

    preds_dict = {}
    for i, user in enumerate(valid_users):
        recommended_indices = Indices[i]
        recommended_item_ids = [idx_to_item_id[idx] for idx in recommended_indices if idx in idx_to_item_id]
        preds_dict[user] = recommended_item_ids

    p, r = calculate_metrics(actual_purchases, preds_dict, k=k)
    print(f"Precision@20 = {p:.5f}")
    print(f"Recall@20    = {r:.5f}")

    return p, r

p_val_nn, r_val_nn = evaluate_semantic_search(val_data, "Validation")
p_test_nn, r_test_nn = evaluate_semantic_search(test_data, "Test")

Precision@20 = 0.00696
Recall@20    = 0.07072
Precision@20 = 0.00352
Recall@20    = 0.03250


In [18]:
BERT_metrics = {
    "BERT": {
        "val_precision_20": float(p_val_nn),
        "val_recall_20": float(r_val_nn),
        "test_precision_20": float(p_test_nn),
        "test_recall_20": float(r_test_nn),
    }
}

metrics_path = ARTIFACTS_DIR / "BERT_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(BERT_metrics, f, indent=4)

In [19]:
train_texts_history = train_data.merge(articles[['article_id', 'text_description']], on='article_id', how='inner')
user_texts = train_texts_history.groupby('customer_id')['text_description'].apply(list)
user_texts = user_texts[user_texts.apply(len) >= 2]

all_texts = articles['text_description'].tolist()
pos_pairs, neg_pairs = [], []

for texts in tqdm(user_texts.sample(10000, random_state=42), desc="Создание пар"):
    t1, t2 = random.sample(texts, 2)
    pos_pairs.append((t1, t2, 1.0))

    t_rand = random.choice(all_texts)
    neg_pairs.append((t1, t_rand, -1.0))

train_pairs = pos_pairs + neg_pairs
random.shuffle(train_pairs)

Создание пар: 100%|██████████| 10000/10000 [00:00<00:00, 342029.19it/s]


In [20]:
class SiameseDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_len=128):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        text_a, text_b, label = self.pairs[idx]

        enc_a = self.tokenizer(text_a, padding='max_length', truncation=True, max_length=self.max_len, return_tensors='pt')
        enc_b = self.tokenizer(text_b, padding='max_length', truncation=True, max_length=self.max_len, return_tensors='pt')

        return {
            'input_ids_a': enc_a['input_ids'].squeeze(0),
            'attention_mask_a': enc_a['attention_mask'].squeeze(0),
            'input_ids_b': enc_b['input_ids'].squeeze(0),
            'attention_mask_b': enc_b['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.float32)
        }

In [21]:
train_dataset = SiameseDataset(train_pairs, tokenizer)
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [22]:
model.train()
optimizer = AdamW(model.parameters(), lr=2e-5)
loss_fn = nn.CosineEmbeddingLoss(margin=0.5)

EPOCHS = 2

for epoch in range(EPOCHS):
    total_loss = 0
    progress_bar = tqdm(train_dataloader, desc=f"Эпоха {epoch+1}/{EPOCHS}")

    for batch in progress_bar:
        input_ids_a = batch['input_ids_a'].to(device)
        mask_a = batch['attention_mask_a'].to(device)
        input_ids_b = batch['input_ids_b'].to(device)
        mask_b = batch['attention_mask_b'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()

        out_a = model(input_ids=input_ids_a, attention_mask=mask_a)
        vec_a = out_a.last_hidden_state[:, 0, :]

        out_b = model(input_ids=input_ids_b, attention_mask=mask_b)
        vec_b = out_b.last_hidden_state[:, 0, :]

        loss = loss_fn(vec_a, vec_b, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})

    avg_loss = total_loss / len(train_dataloader)


Эпоха 2/2: 100%|██████████| 313/313 [01:08<00:00,  4.55it/s, loss=0.1660]


In [23]:
save_path = ARTIFACTS_DIR / "rubert_hm_finetuned"
os.makedirs(save_path, exist_ok=True)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('../artifacts/rubert_hm_finetuned/tokenizer_config.json',
 '../artifacts/rubert_hm_finetuned/tokenizer.json')

In [24]:
model.eval()

article_texts = articles['text_description'].tolist()

with torch.no_grad():
    item_embeddings = extract_bert_embeddings(article_texts, batch_size=512)

BERT Inference: 100%|██████████| 58/58 [00:18<00:00,  3.20it/s]


In [25]:
dimension = item_embeddings.shape[1]

item_embeddings_norm = item_embeddings.copy()
faiss.normalize_L2(item_embeddings_norm)

index = faiss.IndexFlatIP(dimension)
index.add(item_embeddings_norm)

item_id_to_idx = {idx: i for i, idx in enumerate(articles['article_id'].values)}
idx_to_item_id = {i: idx for i, idx in enumerate(articles['article_id'].values)}

In [26]:
p_val_nn, r_val_nn = evaluate_semantic_search(val_data, "Validation")
p_test_nn, r_test_nn = evaluate_semantic_search(test_data, "Test")

Precision@20 = 0.00501
Recall@20    = 0.05129
Precision@20 = 0.00268
Recall@20    = 0.02475


In [27]:
finetuned_metrics = {
    "BERT_finetune": {
        "val_precision_20": float(p_val_nn),
        "val_recall_20": float(r_val_nn),
        "test_precision_20": float(p_test_nn),
        "test_recall_20": float(r_test_nn),
    }
}

with open(ARTIFACTS_DIR / "BERT_finetuned_metrics.json", "w") as f:
    json.dump(finetuned_metrics, f, indent=4)